## Setup and Imports

Load required libraries and define file paths used to build the feature store.

In [2]:
import os
import pandas as pd
import numpy as np

## Define Paths

Set raw and processed data paths and ensure the output directory exists.

In [3]:
raw_data_path = "/Users/paigeblackstone/Library/Mobile Documents/com~apple~CloudDocs/Formula 1 Fun"
processed_path = "data_processed"

os.makedirs(processed_path, exist_ok=True)

## Load Raw Tables

Load all CSV files from the raw data directory into a dictionary of DataFrames.

In [14]:
tables = {}

for file in os.listdir(raw_data_path):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        tables[name] = pd.read_csv(os.path.join(raw_data_path, file))

sorted(tables.keys())

/var/folders/cg/wc2pq_yj52nbzwtdflk50zzc0000gn/T/ipykernel_99087/3846728530.py:6: DtypeWarning: Columns (0: number_x, 1: rank, 2: number_y) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[name] = pd.read_csv(os.path.join(raw_data_path, file))


['circuits',
 'constructor_results',
 'constructor_standings',
 'constructors',
 'driver_standings',
 'drivers',
 'f1_master_enriched',
 'lap_times',
 'pit_stops',
 'qualifying',
 'races',
 'results',
 'seasons',
 'sprint_results',
 'status',
 'weather_data']

## Build Core Master Table

Create the base race-driver-level dataset by merging race results with race, driver, constructor, circuit, and status metadata.

In [15]:
results = tables["results"].copy()
races = tables["races"].copy()
drivers = tables["drivers"].copy()
constructors = tables["constructors"].copy()
circuits = tables["circuits"].copy()
status = tables["status"].copy()

for df in [races, drivers, constructors, circuits]:
    if "url" in df.columns:
        df.drop(columns=["url"], inplace=True)

races["date"] = pd.to_datetime(races["date"], errors="coerce")

master = (
    results
    .merge(races, on="raceId", how="left", validate="m:1")
    .merge(drivers, on="driverId", how="left", validate="m:1")
    .merge(constructors, on="constructorId", how="left", validate="m:1")
    .merge(circuits, on="circuitId", how="left", validate="m:1")
    .merge(status, on="statusId", how="left", validate="m:1")
)

master.shape

(26759, 52)

## Merge Weather Data

Merge race-level historical weather data into the core master table.

In [16]:
weather_path = os.path.join(processed_path, "weather_data.csv")
weather_df = pd.read_csv(weather_path)

master = master.merge(
    weather_df.drop(columns=["race_name", "year"], errors="ignore"),
    on="raceId",
    how="left"
)

master.shape

(26759, 58)

## Geography and Environmental Features

Create derived geographic and environmental variables based on circuit location and weather conditions.

In [17]:
master["month"] = pd.to_datetime(master["date"], errors="coerce").dt.month
master["abs_lat"] = pd.to_numeric(master["lat"], errors="coerce").abs()
master["temp_range"] = (
    pd.to_numeric(master["temp_max"], errors="coerce") -
    pd.to_numeric(master["temp_min"], errors="coerce")
)
master["is_wet_race"] = (pd.to_numeric(master["precipitation"], errors="coerce") > 0).astype(int)
master["high_altitude_track"] = (pd.to_numeric(master["alt"], errors="coerce") >= 500).astype(int)

## Qualifying Features

Merge qualifying data and create summary indicators based on qualifying session progression and pace.

In [18]:
qualifying = tables["qualifying"].copy()
qualifying = qualifying.rename(columns={"position": "qualifying_position"})

for col in ["q1", "q2", "q3"]:
    qualifying[col] = pd.to_timedelta(qualifying[col], errors="coerce")

qualifying["best_qualifying_time_ms"] = (
    qualifying[["q1", "q2", "q3"]]
    .min(axis=1)
    .dt.total_seconds() * 1000
)

qualifying["made_q2"] = qualifying["q2"].notna().astype(int)
qualifying["made_q3"] = qualifying["q3"].notna().astype(int)

master = master.merge(
    qualifying[
        [
            "raceId", "driverId", "qualifying_position",
            "best_qualifying_time_ms", "made_q2", "made_q3"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

master.shape

(26759, 67)

## Sprint Features

Merge sprint race information where available to add sprint-era performance variables.

In [19]:
sprint_results = tables["sprint_results"].copy()

sprint_keep = [
    "raceId", "driverId", "constructorId",
    "grid", "positionOrder", "points", "laps", "milliseconds", "statusId"
]

sprint_results = sprint_results[sprint_keep].rename(columns={
    "grid": "sprint_grid",
    "positionOrder": "sprint_position_order",
    "points": "sprint_points",
    "laps": "sprint_laps",
    "milliseconds": "sprint_milliseconds",
    "statusId": "sprint_status_id"
})

master = master.merge(
    sprint_results,
    on=["raceId", "driverId", "constructorId"],
    how="left"
)

master.shape

(26759, 73)

## Pit Stop Aggregates

Aggregate pit stop records to the race-driver level to create strategy-related features.

In [20]:
pit_stops = tables["pit_stops"].copy()

pit_stops["milliseconds"] = pd.to_numeric(pit_stops["milliseconds"], errors="coerce")
pit_stops["lap"] = pd.to_numeric(pit_stops["lap"], errors="coerce")

pit_agg = (
    pit_stops.groupby(["raceId", "driverId"])
    .agg(
        pit_stop_count=("stop", "count"),
        avg_pit_stop_ms=("milliseconds", "mean"),
        min_pit_stop_ms=("milliseconds", "min"),
        max_pit_stop_ms=("milliseconds", "max"),
        total_pit_stop_ms=("milliseconds", "sum"),
        first_pit_lap=("lap", "min"),
        last_pit_lap=("lap", "max"),
    )
    .reset_index()
)

master = master.merge(
    pit_agg,
    on=["raceId", "driverId"],
    how="left"
)

master.shape

(26759, 80)

## Lap Time Aggregates

Aggregate lap-level timing data to the race-driver level to create pace and consistency variables.

In [21]:
lap_times = tables["lap_times"].copy()

lap_times["milliseconds"] = pd.to_numeric(lap_times["milliseconds"], errors="coerce")
lap_times["lap"] = pd.to_numeric(lap_times["lap"], errors="coerce")

lap_agg = (
    lap_times.groupby(["raceId", "driverId"])
    .agg(
        lap_count_recorded=("lap", "count"),
        avg_lap_time_ms=("milliseconds", "mean"),
        median_lap_time_ms=("milliseconds", "median"),
        std_lap_time_ms=("milliseconds", "std"),
        fastest_lap_in_race_ms=("milliseconds", "min"),
        slowest_lap_ms=("milliseconds", "max"),
    )
    .reset_index()
)

master = master.merge(
    lap_agg,
    on=["raceId", "driverId"],
    how="left"
)

master.shape

(26759, 86)

## Targets and Core Cleaning

Create reusable target variables and standardize core numeric fields.

In [22]:
master["target_points"] = (pd.to_numeric(master["points"], errors="coerce") > 0).astype(int)
master["finish_position"] = pd.to_numeric(master["positionOrder"], errors="coerce")

master["grid_clean"] = pd.to_numeric(master["grid"], errors="coerce")
master.loc[master["grid_clean"] == 0, "grid_clean"] = np.nan

master["points"] = pd.to_numeric(master["points"], errors="coerce")
master["positionOrder"] = pd.to_numeric(master["positionOrder"], errors="coerce")
master["laps"] = pd.to_numeric(master["laps"], errors="coerce")
master["milliseconds"] = pd.to_numeric(master["milliseconds"], errors="coerce")
master["year"] = pd.to_numeric(master["year"], errors="coerce")

## DNF Indicator

Create a reusable binary non-finish indicator based on race status.

In [23]:
finish_like = [
    "Finished",
    "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps",
    "+5 Laps", "+6 Laps", "+7 Laps", "+8 Laps", "+9 Laps"
]

master["is_dnf"] = (~master["status"].isin(finish_like)).astype(int)

## Rolling Driver Features

Create lagged rolling driver-level performance and reliability features using prior races only.

In [24]:
driver_sorted = master.sort_values(["driverId", "date"]).copy()

master["driver_avg_finish_last5"] = (
    driver_sorted.groupby("driverId")["positionOrder"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

master["driver_points_last5"] = (
    driver_sorted.groupby("driverId")["points"]
    .transform(lambda x: x.shift(1).rolling(5).sum())
)

master["driver_dnf_rate_last5"] = (
    driver_sorted.groupby("driverId")["is_dnf"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

master["driver_avg_grid_last5"] = (
    driver_sorted.groupby("driverId")["grid_clean"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

## Rolling Constructor Features

Create lagged rolling constructor-level performance and reliability features using prior races only.

In [25]:
constructor_sorted = master.sort_values(["constructorId", "date"]).copy()

master["constructor_points_last5"] = (
    constructor_sorted.groupby("constructorId")["points"]
    .transform(lambda x: x.shift(1).rolling(5).sum())
)

master["constructor_dnf_rate_last5"] = (
    constructor_sorted.groupby("constructorId")["is_dnf"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

master["constructor_avg_finish_last5"] = (
    constructor_sorted.groupby("constructorId")["positionOrder"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

## Lagged Standings Features

Create pre-race driver and constructor standings variables by shifting standings history forward one race.

In [26]:
driver_standings = tables["driver_standings"].copy()
constructor_standings = tables["constructor_standings"].copy()

driver_standings["points"] = pd.to_numeric(driver_standings["points"], errors="coerce")
driver_standings["position"] = pd.to_numeric(driver_standings["position"], errors="coerce")
driver_standings["wins"] = pd.to_numeric(driver_standings["wins"], errors="coerce")

constructor_standings["points"] = pd.to_numeric(constructor_standings["points"], errors="coerce")
constructor_standings["position"] = pd.to_numeric(constructor_standings["position"], errors="coerce")
constructor_standings["wins"] = pd.to_numeric(constructor_standings["wins"], errors="coerce")

# attach race dates so standings can be sorted in race order
driver_standings = driver_standings.merge(
    races[["raceId", "date"]],
    on="raceId",
    how="left"
)

constructor_standings = constructor_standings.merge(
    races[["raceId", "date"]],
    on="raceId",
    how="left"
)

driver_standings = driver_standings.sort_values(["driverId", "date"]).copy()
constructor_standings = constructor_standings.sort_values(["constructorId", "date"]).copy()

driver_standings["driver_standing_points_prerace"] = (
    driver_standings.groupby("driverId")["points"].shift(1)
)
driver_standings["driver_standing_position_prerace"] = (
    driver_standings.groupby("driverId")["position"].shift(1)
)
driver_standings["driver_standing_wins_prerace"] = (
    driver_standings.groupby("driverId")["wins"].shift(1)
)

constructor_standings["constructor_standing_points_prerace"] = (
    constructor_standings.groupby("constructorId")["points"].shift(1)
)
constructor_standings["constructor_standing_position_prerace"] = (
    constructor_standings.groupby("constructorId")["position"].shift(1)
)
constructor_standings["constructor_standing_wins_prerace"] = (
    constructor_standings.groupby("constructorId")["wins"].shift(1)
)

master = master.merge(
    driver_standings[
        [
            "raceId", "driverId",
            "driver_standing_points_prerace",
            "driver_standing_position_prerace",
            "driver_standing_wins_prerace"
        ]
    ],
    on=["raceId", "driverId"],
    how="left"
)

master = master.merge(
    constructor_standings[
        [
            "raceId", "constructorId",
            "constructor_standing_points_prerace",
            "constructor_standing_position_prerace",
            "constructor_standing_wins_prerace"
        ]
    ],
    on=["raceId", "constructorId"],
    how="left"
)

master.shape

(26759, 103)

## Final Type Standardization

Ensure modeling-relevant variables are stored using consistent numeric types.

In [27]:
numeric_cols = [
    "qualifying_position",
    "best_qualifying_time_ms",
    "pit_stop_count",
    "avg_pit_stop_ms",
    "min_pit_stop_ms",
    "max_pit_stop_ms",
    "total_pit_stop_ms",
    "first_pit_lap",
    "last_pit_lap",
    "lap_count_recorded",
    "avg_lap_time_ms",
    "median_lap_time_ms",
    "std_lap_time_ms",
    "fastest_lap_in_race_ms",
    "slowest_lap_ms",
    "sprint_grid",
    "sprint_position_order",
    "sprint_points",
    "sprint_laps",
    "sprint_milliseconds",
    "temp_avg",
    "temp_max",
    "temp_min",
    "precipitation",
    "wind_speed",
    "humidity_avg",
    "abs_lat",
    "temp_range",
    "driver_avg_finish_last5",
    "driver_points_last5",
    "driver_dnf_rate_last5",
    "driver_avg_grid_last5",
    "constructor_points_last5",
    "constructor_dnf_rate_last5",
    "constructor_avg_finish_last5",
    "driver_standing_points_prerace",
    "driver_standing_position_prerace",
    "driver_standing_wins_prerace",
    "constructor_standing_points_prerace",
    "constructor_standing_position_prerace",
    "constructor_standing_wins_prerace",
]

for col in numeric_cols:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce")

## Save Feature Store Outputs

Save the complete feature store and specialized subsets for downstream modeling notebooks.

In [28]:
# full master feature store
master.to_csv(os.path.join(processed_path, "f1_feature_store_master.csv"), index=False)
master.to_parquet(os.path.join(processed_path, "f1_feature_store_master.parquet"), index=False)

## Save Pre-Race Feature Store

Create and save a subset containing only features that would be available before race start.

In [29]:
prerace_cols = [
    "raceId", "driverId", "constructorId", "circuitId", "date", "year", "round",
    "target_points", "finish_position", "positionOrder", "points",
    "grid", "grid_clean",
    "qualifying_position", "best_qualifying_time_ms", "made_q2", "made_q3",
    "driver_avg_finish_last5", "driver_points_last5", "driver_dnf_rate_last5", "driver_avg_grid_last5",
    "constructor_points_last5", "constructor_dnf_rate_last5", "constructor_avg_finish_last5",
    "driver_standing_points_prerace", "driver_standing_position_prerace", "driver_standing_wins_prerace",
    "constructor_standing_points_prerace", "constructor_standing_position_prerace", "constructor_standing_wins_prerace",
    "lat", "lng", "alt", "abs_lat",
    "temp_avg", "temp_max", "temp_min", "temp_range",
    "precipitation", "wind_speed", "humidity_avg",
    "month", "is_wet_race", "high_altitude_track",
    "sprint_grid", "sprint_position_order", "sprint_points", "sprint_laps", "sprint_milliseconds"
]

prerace_cols = [c for c in prerace_cols if c in master.columns]

prerace_store = master[prerace_cols].copy()

prerace_store.to_parquet(os.path.join(processed_path, "f1_feature_store_prerace.parquet"), index=False)
prerace_store.to_csv(os.path.join(processed_path, "f1_feature_store_prerace.csv"), index=False)

## Save Strategy Feature Store

Create and save a subset containing in-race strategy and pace variables for race strategy analysis.

In [30]:
strategy_cols = prerace_cols + [
    "pit_stop_count", "avg_pit_stop_ms", "min_pit_stop_ms", "max_pit_stop_ms", "total_pit_stop_ms",
    "first_pit_lap", "last_pit_lap",
    "lap_count_recorded", "avg_lap_time_ms", "median_lap_time_ms", "std_lap_time_ms",
    "fastest_lap_in_race_ms", "slowest_lap_ms",
    "milliseconds", "laps", "fastestLap", "fastestLapTime", "fastestLapSpeed"
]

strategy_cols = [c for c in strategy_cols if c in master.columns]

strategy_store = master[strategy_cols].copy()

strategy_store.to_parquet(os.path.join(processed_path, "f1_feature_store_strategy.parquet"), index=False)
strategy_store.to_csv(os.path.join(processed_path, "f1_feature_store_strategy.csv"), index=False)

## Final Validation

Inspect the shapes and a sample of each saved feature store.

In [31]:
print("Master:", master.shape)
print("Pre-race:", prerace_store.shape)
print("Strategy:", strategy_store.shape)

master.head()

Master: (26759, 103)
Pre-race: (26759, 49)
Strategy: (26759, 67)


,resultId,raceId,driverId,constructorId,number_x,grid,position,positionText,positionOrder,points,...,driver_avg_grid_last5,constructor_points_last5,constructor_dnf_rate_last5,constructor_avg_finish_last5,driver_standing_points_prerace,driver_standing_position_prerace,driver_standing_wins_prerace,constructor_standing_points_prerace,constructor_standing_position_prerace,constructor_standing_wins_prerace
0,1,18,1,1,22,1,1,1,1,10.0,...,2.0,16.0,0.4,10.4,109.0,2.0,4.0,218.0,11.0,8.0
1,2,18,2,2,3,5,2,2,2,8.0,...,5.8,9.0,0.4,10.0,61.0,5.0,0.0,101.0,2.0,0.0
2,3,18,3,3,7,7,3,3,3,6.0,...,10.8,5.0,0.2,12.8,20.0,9.0,0.0,33.0,4.0,0.0
3,4,18,4,4,5,11,4,4,4,5.0,...,2.8,4.0,0.4,12.8,109.0,3.0,4.0,51.0,3.0,0.0
4,5,18,5,1,23,3,5,5,5,4.0,...,11.4,26.0,0.2,6.4,30.0,7.0,0.0,218.0,11.0,8.0


## Save Outputs to Multiple Locations

Save feature-store outputs both to the project repository and to the external data directory for backup and reuse.

In [32]:
import os

# repo-local output
repo_processed_path = "data_processed"

# external backup/output
external_processed_path = "/Users/paigeblackstone/Library/Mobile Documents/com~apple~CloudDocs/Formula 1 Fun/processed_outputs"

os.makedirs(repo_processed_path, exist_ok=True)
os.makedirs(external_processed_path, exist_ok=True)

# objects to save
datasets_to_save = {
    "f1_feature_store_master": master,
    "f1_feature_store_prerace": prerace_store,
    "f1_feature_store_strategy": strategy_store,
    "weather_data": weather_df
}

for name, dataset in datasets_to_save.items():
    # save to repo
    dataset.to_csv(os.path.join(repo_processed_path, f"{name}.csv"), index=False)
    dataset.to_parquet(os.path.join(repo_processed_path, f"{name}.parquet"), index=False)

    # save to external backup
    dataset.to_csv(os.path.join(external_processed_path, f"{name}.csv"), index=False)
    dataset.to_parquet(os.path.join(external_processed_path, f"{name}.parquet"), index=False)

print("Saved files to:")
print(" -", os.path.abspath(repo_processed_path))
print(" -", external_processed_path)

Saved files to:
 - /Users/paigeblackstone/Documents/GitHub/f1-worldchamp/data_processed
 - /Users/paigeblackstone/Library/Mobile Documents/com~apple~CloudDocs/Formula 1 Fun/processed_outputs


In [33]:
print("\nRepo files:")
print(sorted(os.listdir(repo_processed_path)))

print("\nExternal files:")
print(sorted(os.listdir(external_processed_path)))


Repo files:
['f1_feature_store.csv', 'f1_feature_store.parquet', 'f1_feature_store_enriched_pitstops.parquet', 'f1_feature_store_master.csv', 'f1_feature_store_master.parquet', 'f1_feature_store_prerace.csv', 'f1_feature_store_prerace.parquet', 'f1_feature_store_strategy.csv', 'f1_feature_store_strategy.parquet', 'f1_master_enriched.csv', 'weather_data.csv', 'weather_data.parquet']

External files:
['f1_feature_store_master.csv', 'f1_feature_store_master.parquet', 'f1_feature_store_prerace.csv', 'f1_feature_store_prerace.parquet', 'f1_feature_store_strategy.csv', 'f1_feature_store_strategy.parquet', 'weather_data.csv', 'weather_data.parquet']
